In [ ]:
# IMPORT REQUIRED LIBRARIES
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

#  DEFINE THE API EXTRACTOR
def fetch_academic_papers():
    print("Connecting to OpenAlex API...")
    url = "https://api.openalex.org/works"
    params = {
        "filter": "institutions.country_code:nl|fi,publication_year:>2023",
        "search": "computer science OR data analysis",
        "per_page": 100,
        "mailto": "antlasarfaraz19@gmail.com"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        print(f"Successfully retrieved {len(data.get('results', []))} recent research papers.")
        return data.get('results', [])
    else:
        print(f"Failed to fetch data: HTTP {response.status_code}")
        return []

def rebuild_abstract(inverted_index):
    if not inverted_index:
        return ""
    max_idx = max([idx for indices in inverted_index.values() for idx in indices])
    words = [""] * (max_idx + 1)
    for word, indices in inverted_index.items():
        for idx in indices:
            words[idx] = word
    return " ".join(words).strip()


In [ ]:
#  EXECUTE EXTRACTION & FLATTEN JSON
raw_papers = fetch_academic_papers()

# CRITICAL FIX: Stop the script gracefully if the API fails, preventing KeyErrors
if not raw_papers:
    print("Pipeline stopped: No data was retrieved from the API. Please try again in a few minutes.")
else:
    paper_records = []
    for paper in raw_papers:
        title = paper.get('title', 'No Title')
        year = paper.get('publication_year', None)

        authors = paper.get('authorships', [])
        country = "Unknown"
        if authors and authors[0].get('institutions'):
            country = authors[0]['institutions'][0].get('country_code', 'Unknown')

        raw_abstract = paper.get('abstract_inverted_index', {})
        clean_abstract = rebuild_abstract(raw_abstract)

        paper_records.append({
            "Title": title,
            "Year": year,
            "Country": country,
            "Abstract": clean_abstract
        })

    df_papers = pd.DataFrame(paper_records)

    # Ensure the dataframe actually has the 'Abstract' column before filtering
    if 'Abstract' in df_papers.columns:
        df_papers = df_papers[df_papers['Abstract'] != ""]

In [ ]:
#  NATURAL LANGUAGE PROCESSING (NLP) TEXT CLEANING
    print("Processing and cleaning abstract text...")

    stop_words = set([
        "the", "and", "of", "to", "in", "a", "is", "that", "for", "we", "this",
        "with", "on", "as", "are", "by", "an", "be", "can", "it", "from", "which",
        "our", "these", "has", "have", "been", "was", "were", "their", "model",
        "data", "paper", "results", "using", "study", "based", "proposed", "also"
    ])

    def extract_keywords(text):
        text = str(text).lower()
        words = re.findall(r'\b[a-z]{3,}\b', text)
        return [word for word in words if word not in stop_words]

    all_keywords = df_papers['Abstract'].apply(extract_keywords).sum()
    word_counts = Counter(all_keywords)
    top_words = word_counts.most_common(15)
    df_top_words = pd.DataFrame(top_words, columns=['Keyword', 'Frequency'])

    df_papers.to_csv("dutch_finnish_cs_research_trends.csv", index=False)
    print("Data saved to 'dutch_finnish_cs_research_trends.csv'.")


In [ ]:
# VISUALIZE THE RESEARCH TRENDS
    plt.figure(figsize=(12, 6))
    sns.set_theme(style="whitegrid")
    sns.barplot(data=df_top_words, x='Frequency', y='Keyword', palette='mako')
    plt.title('Top Trending Keywords in Recent Dutch & Finnish Computer Science Research', fontsize=14, pad=15)
    plt.xlabel('Frequency of Mention in Abstracts', fontsize=12)
    plt.ylabel('Research Keyword', fontsize=12)
    plt.tight_layout()
    plt.show()

    display(df_papers[['Title', 'Country']].head())